In [ ]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../../data/processed/delivery_clean.parquet")

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(df['accept_gps_lat'], df['accept_gps_lng'],
                               df['delivery_gps_lat'], df['delivery_gps_lng'])
df['accept_hour'] = df['accept_time'].dt.hour

df['is_instant_delivery'] = (df['delivery_duration_min'] == 0).astype(int)
df['is_ghost_dispatch'] = df['accept_gps_lng'].isnull().astype(int)

duration_hours = df['delivery_duration_min'] / 60.0
df['implied_speed_kmh'] = np.where(
    df['delivery_duration_min'] > 0,
    df['distance_km'] / duration_hours,
    0.0
)

EPS = 1e-5

dur_stats = df.groupby('city')['delivery_duration_min'].agg(
    dur_med='median', dur_q1=lambda x: x.quantile(.25), dur_q3=lambda x: x.quantile(.75))
dur_stats['dur_iqr'] = dur_stats['dur_q3'] - dur_stats['dur_q1']
df = df.merge(dur_stats[['dur_med','dur_iqr']], on='city', how='left')
df['duration_robust_city'] = (df['delivery_duration_min'] - df['dur_med']) / (df['dur_iqr'] + EPS)

dist_stats = df.groupby('city')['distance_km'].agg(
    dist_med='median', dist_q1=lambda x: x.quantile(.25), dist_q3=lambda x: x.quantile(.75))
dist_stats['dist_iqr'] = dist_stats['dist_q3'] - dist_stats['dist_q1']
df = df.merge(dist_stats[['dist_med','dist_iqr']], on='city', how='left')
df['distance_robust_city'] = (df['distance_km'] - df['dist_med']) / (df['dist_iqr'] + EPS)

IF_FEATURES = ['duration_robust_city','distance_robust_city','implied_speed_kmh',
               'accept_hour','is_instant_delivery','is_ghost_dispatch','aoi_type']
print("Feature build complete. Sanity check for inf/NaN:")
for c in IF_FEATURES:
    n_inf = np.isinf(df[c]).sum() if df[c].dtype.kind=='f' else 0
    n_nan = df[c].isnull().sum()
    print(f"  {c:24s} inf={n_inf:>6}  nan={n_nan:>6}")

Feature build complete. Sanity check for inf/NaN:
  duration_robust_city     inf=     0  nan=     0
  distance_robust_city     inf=     0  nan=  3375
  implied_speed_kmh        inf=     0  nan=  3308
  accept_hour              inf=     0  nan=     0
  is_instant_delivery      inf=     0  nan=     0
  is_ghost_dispatch        inf=     0  nan=     0
  aoi_type                 inf=     0  nan=     0


In [ ]:
df['distance_robust_city'] = df['distance_robust_city'].fillna(0)
df['distance_km'] = df['distance_km'].fillna(0)
df['implied_speed_kmh'] = df['implied_speed_kmh'].fillna(0)

IF_FEATURES = ['duration_robust_city','distance_robust_city','implied_speed_kmh',
               'accept_hour','is_instant_delivery','is_ghost_dispatch','aoi_type']
print("After fill — final check:")
clean = True
for c in IF_FEATURES:
    n_inf = np.isinf(df[c]).sum() if df[c].dtype.kind=='f' else 0
    n_nan = df[c].isnull().sum()
    if n_inf or n_nan: clean = False
    print(f"  {c:24s} inf={n_inf:>6}  nan={n_nan:>6}")
print(f"\n{'ALL CLEAN — ready for IsolationForest' if clean else 'STILL HAS ISSUES'}")

After fill — final check:
  duration_robust_city     inf=     0  nan=     0
  distance_robust_city     inf=     0  nan=     0
  implied_speed_kmh        inf=     0  nan=     0
  accept_hour              inf=     0  nan=     0
  is_instant_delivery      inf=     0  nan=     0
  is_ghost_dispatch        inf=     0  nan=     0
  aoi_type                 inf=     0  nan=     0

ALL CLEAN — ready for IsolationForest


In [ ]:
KEEP = ['order_id','city','courier_id','ds','aoi_id',
        'delivery_duration_min','distance_km', 
        'duration_robust_city','distance_robust_city','implied_speed_kmh',
        'accept_hour','is_instant_delivery','is_ghost_dispatch','aoi_type']

df_feat = df[KEEP].copy()
OUT = "../../data/processed/delivery_features.parquet"
df_feat.to_parquet(OUT, index=False)
print(f"Saved {len(df_feat):,} rows → {OUT}")
print(f"Columns: {list(df_feat.columns)}")
print(f"\nIF feature set (7): {IF_FEATURES}")

Saved 4,512,573 rows → ../../data/processed/delivery_features.parquet
Columns: ['order_id', 'city', 'courier_id', 'ds', 'aoi_id', 'delivery_duration_min', 'distance_km', 'duration_robust_city', 'distance_robust_city', 'implied_speed_kmh', 'accept_hour', 'is_instant_delivery', 'is_ghost_dispatch', 'aoi_type']

IF feature set (7): ['duration_robust_city', 'distance_robust_city', 'implied_speed_kmh', 'accept_hour', 'is_instant_delivery', 'is_ghost_dispatch', 'aoi_type']
